## Data Preprocessing

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load the dataset from Google Sheets
# The URL provided points to a Google Sheet. We need to convert it to an export URL for pandas to read it directly.
google_sheet_url = 'https://docs.google.com/spreadsheets/d/1mnoUgK8YxInzoDQ7P7iBM1HeZ8tcnUSvMU_H1OWndFA/edit?gid=0#gid=0'
csv_export_url = google_sheet_url.replace('/edit?gid=', '/export?format=csv&gid=')

df = pd.read_csv(csv_export_url)

print("Dataset loaded successfully:")
display(df.head())

# Separate input features (X) from the target variable (y)
X = df[['Outlook', 'Temperature', 'Humidity', 'Wind']].copy() # Make a copy to avoid SettingWithCopyWarning later
y = df['Play Tennis'] # Corrected column name

print("\nFeatures (X) and Target (y) separated.")

Dataset loaded successfully:


,No,Outlook,Temperature,Humidity,Wind,Play Tennis
0,1,Sunny,Hot,High,Weak,No
1,2,Sunny,Hot,High,Strong,No
2,3,Overcast,Hot,High,Weak,Yes
3,4,Rain,Mild,High,Weak,Yes
4,5,Rain,Cool,Normal,Weak,Yes



Features (X) and Target (y) separated.


This cell **loads the dataset** from a Google Sheet into a pandas DataFrame (`df`) and **separates the input features (X)** from the **target variable (y)**.

In [ ]:
# Convert all categorical feature values and target labels into numerical representations
le = LabelEncoder()

# Apply LabelEncoder to each categorical feature in X
for column in X.columns:
    X.loc[:, column] = le.fit_transform(X[column]) # Use .loc for proper assignment

# Apply LabelEncoder to the target variable y
y = le.fit_transform(y)

print("\nCategorical features and target variable encoded:")
display(pd.DataFrame(X, columns=['Outlook', 'Temperature', 'Humidity', 'Wind']).head())
print(f"Encoded target variable (first 5 values): {y[:5]}")


Categorical features and target variable encoded:


,Outlook,Temperature,Humidity,Wind
0,2,1,0,1
1,2,1,0,0
2,0,1,0,1
3,1,2,0,1
4,1,0,1,1


Encoded target variable (first 5 values): [0 0 1 1 1]


This cell **converts all categorical string values** in features (`X`) and the target (`y`) into numerical representations using `LabelEncoder`. The **result** is `X` and `y` containing integer-encoded values.

## Dataset Partitioning

In [ ]:
from sklearn.model_selection import train_test_split

# Divide the dataset into training and testing subsets using an 80:20 train-test split ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print("Dataset successfully split into training and testing subsets.")

Training set size: 40 samples
Test set size: 10 samples
Dataset successfully split into training and testing subsets.


This cell **splits the preprocessed data** (`X`, `y`) into training (`X_train`, `y_train`) and testing (`X_test`, `y_test`) sets.

## Naive Bayes Model Training & Evaluation

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize and train the Categorical Naive Bayes model
cnb = CategoricalNB()
cnb.fit(X_train, y_train)

print("Categorical Naive Bayes model trained successfully.")

# Predict the class labels for the test dataset
y_pred_cnb = cnb.predict(X_test)

# Calculate and display the overall Model Accuracy
accuracy_cnb = accuracy_score(y_test, y_pred_cnb)
print(f"\nModel Accuracy (Categorical Naive Bayes): {accuracy_cnb:.4f}")

# Display the Confusion Matrix
conf_matrix_cnb = confusion_matrix(y_test, y_pred_cnb)
print("\nConfusion Matrix (Categorical Naive Bayes):\n", conf_matrix_cnb)

# Display the Classification Report (Precision, Recall, F1-Score)
class_report_cnb = classification_report(y_test, y_pred_cnb)
print("\nClassification Report (Categorical Naive Bayes):\n", class_report_cnb)

Categorical Naive Bayes model trained successfully.

Model Accuracy (Categorical Naive Bayes): 0.8000

Confusion Matrix (Categorical Naive Bayes):
 [[1 1]
 [1 7]]

Classification Report (Categorical Naive Bayes):
               precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10



This cell **trains a Categorical Naive Bayes model** and **evaluates its performance**. The **result** includes model accuracy, a confusion matrix, and a classification report. This is **done to build a probabilistic classifier** suitable for our categorical data. An accuracy of 0.8000 suggests the model correctly predicts 80% of test samples, while the confusion matrix provides detailed error breakdown.

## Single-Sample Inference

In [ ]:
# Define the single sample for prediction
single_sample = pd.DataFrame([{
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}])

print("Single sample for prediction:")
display(single_sample)

# To perform inference, we need to encode the single sample using the same LabelEncoders
# used for the training data. We need to recreate the LabelEncoders for each feature
# to ensure consistent mapping, as the original 'le' was applied iteratively.

# Re-initialize LabelEncoders for each column based on the original dataframe 'df'
# This is crucial to ensure that the encoding is consistent with the training data.
encoded_single_sample = single_sample.copy()
original_features = df[['Outlook', 'Temperature', 'Humidity', 'Wind']]

for column in original_features.columns:
    le_col = LabelEncoder()
    le_col.fit(original_features[column]) # Fit on original data column
    encoded_single_sample[column] = le_col.transform(single_sample[column])

print("\nEncoded single sample:")
display(encoded_single_sample)

# Predict the class label
predicted_label = cnb.predict(encoded_single_sample)

# Predict the class probabilities
predicted_proba = cnb.predict_proba(encoded_single_sample)

# We need to reverse transform the predicted label to its original categorical form
# For this, we need the LabelEncoder used for the 'Play Tennis' target variable.
# Let's re-fit a LabelEncoder for 'Play Tennis' to ensure we have the inverse mapping.
le_play_tennis = LabelEncoder()
le_play_tennis.fit(df['Play Tennis'])

decoded_label = le_play_tennis.inverse_transform(predicted_label)

print(f"\nPredicted class label: {decoded_label[0]}")
print(f"Class probabilities (No, Yes): {predicted_proba[0]}")

Single sample for prediction:


,Outlook,Temperature,Humidity,Wind
0,Sunny,Cool,High,Strong



Encoded single sample:


,Outlook,Temperature,Humidity,Wind
0,2,0,0,0



Predicted class label: No
Class probabilities (No, Yes): [0.92560203 0.07439797]


This cell **performs a prediction and probability estimation** for a single new data point using the trained Naive Bayes model. The **result** is the decoded predicted label ('No' or 'Yes') and the probabilities for each class. This is **done to demonstrate** how the trained model handles new, unseen inputs after consistent encoding. The high probability (e.g., 0.9256 for 'No') indicates the model's confidence in the prediction for the given sample.

## Model Comparison

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train a Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

print("Decision Tree Classifier trained successfully.")

# Predict the class labels for the test dataset
y_pred_dt = dt_classifier.predict(X_test)

# Calculate and display the overall Model Accuracy
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f"\nModel Accuracy (Decision Tree): {accuracy_dt:.4f}")

# Predict the class label for the single sample using Decision Tree
predicted_label_dt = dt_classifier.predict(encoded_single_sample)
decoded_label_dt = le_play_tennis.inverse_transform(predicted_label_dt)

# Predict the class probabilities for the single sample using Decision Tree
predicted_proba_dt = dt_classifier.predict_proba(encoded_single_sample)

print(f"\nSingle-sample prediction (Decision Tree): {decoded_label_dt[0]}")
print(f"Class probabilities (Decision Tree - No, Yes): {predicted_proba_dt[0]}")

Decision Tree Classifier trained successfully.

Model Accuracy (Decision Tree): 0.8000

Single-sample prediction (Decision Tree): No
Class probabilities (Decision Tree - No, Yes): [1. 0.]


This cell **trains a Decision Tree Classifier**, **evaluates its accuracy**, and **performs single-sample prediction/probability**. The **result** is the model's accuracy, predicted label, and probabilities for the sample. This is **done to compare** its performance against other models, as Decision Trees capture complex relationships. An accuracy of 0.8000 suggests good performance, with a 100% certainty prediction of 'No' for the specific sample.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train a Logistic Regression Classifier
# For categorical features, Logistic Regression typically expects one-hot encoding.
# Since we used LabelEncoder, it treats them as ordinal. This might not be ideal
# but we will proceed as per the prompt's request to use the same training data.
# We set solver='liblinear' and C=0.1 for better convergence with small datasets.
lr_classifier = LogisticRegression(random_state=42, solver='liblinear', C=0.1)
lr_classifier.fit(X_train, y_train)

print("Logistic Regression Classifier trained successfully.")

# Predict the class labels for the test dataset
y_pred_lr = lr_classifier.predict(X_test)

# Calculate and display the overall Model Accuracy
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"\nModel Accuracy (Logistic Regression): {accuracy_lr:.4f}")

# Predict the class label for the single sample using Logistic Regression
predicted_label_lr = lr_classifier.predict(encoded_single_sample)
decoded_label_lr = le_play_tennis.inverse_transform(predicted_label_lr)

# Predict the class probabilities for the single sample using Logistic Regression
predicted_proba_lr = lr_classifier.predict_proba(encoded_single_sample)

print(f"\nSingle-sample prediction (Logistic Regression): {decoded_label_lr[0]}")
print(f"Class probabilities (Logistic Regression - No, Yes): {predicted_proba_lr[0]}")

Logistic Regression Classifier trained successfully.

Model Accuracy (Logistic Regression): 0.7000

Single-sample prediction (Logistic Regression): No
Class probabilities (Logistic Regression - No, Yes): [0.67111864 0.32888136]


This cell **trains a Logistic Regression Classifier**, **evaluates its accuracy**, and **performs single-sample prediction/probability**. The **result** is the model's accuracy, predicted label, and probabilities. This is **done to benchmark** a linear model, despite `LabelEncoder` treating features as ordinal. An accuracy of 0.7000 indicates slightly lower performance, and the sample prediction 'No' with probabilities [0.6711, 0.3289] shows the model's belief.

In [ ]:
from sklearn.svm import SVC

# Train an SVM Classifier
# SVC does not natively provide probability estimates unless probability=True is set,
# which uses cross-validation and can be computationally expensive.
# For consistency with other models, we'll enable it for single-sample probability.
svm_classifier = SVC(probability=True, random_state=42)
svm_classifier.fit(X_train, y_train)

print("SVM Classifier trained successfully.")

# Predict the class labels for the test dataset
y_pred_svm = svm_classifier.predict(X_test)

# Calculate and display the overall Model Accuracy
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"\nModel Accuracy (SVM): {accuracy_svm:.4f}")

# Predict the class label for the single sample using SVM
predicted_label_svm = svm_classifier.predict(encoded_single_sample)
decoded_label_svm = le_play_tennis.inverse_transform(predicted_label_svm)

# Predict the class probabilities for the single sample using SVM
predicted_proba_svm = svm_classifier.predict_proba(encoded_single_sample)

print(f"\nSingle-sample prediction (SVM): {decoded_label_svm[0]}")
print(f"Class probabilities (SVM - No, Yes): {predicted_proba_svm[0]}")

SVM Classifier trained successfully.

Model Accuracy (SVM): 0.7000

Single-sample prediction (SVM): No
Class probabilities (SVM - No, Yes): [0.97514137 0.02485863]


This cell **trains a Support Vector Machine (SVM) Classifier**, **evaluates its accuracy**, and **performs single-sample prediction/probability**. The **result** is the model's accuracy, predicted label, and probabilities. This is **done to utilize a powerful classification algorithm** for comparison, with `probability=True` for estimates. An accuracy of 0.7000 is similar to Logistic Regression, and a high probability (0.9751) for 'No' indicates SVM's confidence for the sample.

In [ ]:
  # Prepare data for comparison table
comparison_data = {
    'Model': ['Categorical Naive Bayes', 'Decision Tree', 'Logistic Regression', 'SVM'],
    'Test Accuracy': [accuracy_cnb, accuracy_dt, accuracy_lr, accuracy_svm],
    'Single Sample Prediction': [decoded_label[0], decoded_label_dt[0], decoded_label_lr[0], decoded_label_svm[0]],
    'Single Sample Probabilities (No)': [predicted_proba[0][0], predicted_proba_dt[0][0], predicted_proba_lr[0][0], predicted_proba_svm[0][0]],
    'Single Sample Probabilities (Yes)': [predicted_proba[0][1], predicted_proba_dt[0][1], predicted_proba_lr[0][1], predicted_proba_svm[0][1]]
}

comparison_df = pd.DataFrame(comparison_data)

print("\nModel Comparison Summary:")
display(comparison_df)


Model Comparison Summary:


,Model,Test Accuracy,Single Sample Prediction,Single Sample Probabilities (No),Single Sample Probabilities (Yes)
0,Categorical Naive Bayes,0.8,No,0.925602,0.074398
1,Decision Tree,0.8,No,1.000000,0.000000
2,Logistic Regression,0.7,No,0.671119,0.328881
3,SVM,0.7,No,0.975141,0.024859


This cell **consolidates and displays key evaluation metrics and single-sample prediction results** from all trained models. The **result** is a comparative pandas DataFrame. This is **done to provide a clear, side-by-side comparison** of model performance. The table allows quick assessment, showing Naive Bayes and Decision Tree as top performers (0.8000 accuracy) and consistent 'No' predictions with varying confidence levels for the sample.

## Overall Conclusion

This notebook **explores a machine learning pipeline** by loading, preprocessing, splitting, training, and evaluating four classification models on a categorical dataset. The **result** showed varied accuracies (Naive Bayes/Decision Tree: 0.80, Logistic Regression/SVM: 0.70) and consistent 'No' predictions for a single sample with different confidence levels. This was **done to compare algorithm suitability** and understand prediction behavior across models. The **interpretation** is that Naive Bayes and Decision Tree were better suited to this specific dataset, and all models strongly agreed on the 'No' prediction for the given sample.